# 모델 만들기

당일 **장 시작(09:30) 직전까지** 쓸 수 있는 데이터로 그날 종가 방향을 맞히는 문제임.

| | |
|---|---|
| 0 | 급하락 — −2.5% 미만 |
| 1 | 하락 — −2.5% ~ −0.6% |
| 2 | 보합 — −0.6% ~ +0.6% |
| 3 | 상승 — +0.6% ~ +2.5% |
| 4 | 급상승 — +2.5% 초과 |

위에서부터 차례로 따라가면 됨.
데이터를 먼저 둘러보려면 `1_eda.ipynb` 부터 보면 됨.

1. 데이터 둘러보기
2. 기간 나누기 — holdout · walk-forward · rolling
3. 기술지표로 feature 만들기
4. 기준선 — 어제 label 따라가기
5. 랜덤 포레스트
6. LSTM 한 층
7. `model.py` 채우기

In [1]:
import sys, warnings
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == 'example' else Path.cwd()))
warnings.filterwarnings('ignore')

from src import (Dataset, evaluate, predict, check, report, LABELS,
                 RET_CUTS, ARTIFACTS, CRASH, DOWN, FLAT, UP, SURGE)
from src.features import rsi, macd, bollinger

ds = Dataset()
ds

<Dataset 50종목 기준일 2024-08-13~2026-09-11 522일 (일봉 2023-10-23~2026-09-14)>

## 1. 데이터 둘러보기

`ds.day(기준일)` 로 하루를 잡음. `day.date` 는 **기준일**, `day.target` 은 **예측 대상일(다음 거래일)**,
`day.cutoff` 는 대상일 09:30 임. 여기서 뭘 꺼내든 **전부 cutoff 에서 잘림.**
기준일 장 마감 후 뉴스와 대상일 개장 전 시간외 거래까지는 쓸 수 있음.

In [2]:
day = ds.day('2026-09-01')
print(day, '| 예측할 종목', len(day.symbols))

day.daily(days=10).tail(3)

<Day 2026-09-01 -> 2026-09-02 cutoff 09-02 09:30> | 예측할 종목 50


,symbol,date_et,open,high,low,close,volume,prev_close,ret,up,known_at
497,IBKR,2026-09-01,94.625,94.900002,90.025002,90.419998,4419303,97.279999,-0.070518,0,2026-09-01 16:00:00
498,COP,2026-09-01,135.000,136.300003,133.637299,136.250000,5279305,132.500000,0.028302,1,2026-09-01 16:00:00
499,LLY,2026-09-01,1188.000,1188.324951,1156.850098,1160.000000,1730004,1157.505005,0.002155,1,2026-09-01 16:00:00


In [3]:
# 구간은 hours= / days= / since= 중 하나로 줌. 안 주면 처음부터임.
print('일봉   ', len(day.daily(days=120)))
print('시간봉 ', len(day.price(hours=18)))
print('기사   ', len(day.news(hours=24)))
print('실적   ', len(day.earnings(days=90)))
print('목표주가', len(day.analyst(days=30)))
print('Reddit   ', len(day.reddit('stocks', hours=24)))

일봉    6000
시간봉  547
기사    3941
실적    79
목표주가 417


Reddit    1690


In [4]:
# 대상일 시간봉은 pre 세션만 나옴. 정규장 봉은 09:30 이후에 닫히니 안 보임.
p = day.price(hours=18)
print(p.session.value_counts().to_dict())
print('기준일 봉:', sorted(set(p[p.datetime.dt.normalize() == day.date].session)))
print('대상일 봉:', sorted(set(p[p.datetime.dt.normalize() == day.target].session)))

{'pre': 248, 'post': 199, 'regular': 100}
기준일 봉: ['post', 'regular']
대상일 봉: ['pre']


### 미래를 보지 않음

자르는 기준이 `known_at` 인데, **언제 일어났나**가 아니라 **언제 알 수 있었나**임.

| 데이터 | 일어난 시각 | 알 수 있게 된 시각 |
|---|---|---|
| 1시간봉 | 봉의 **시작** (09:30) | 봉이 닫히는 10:30 |
| 일봉 | 그날 | 16:00 마감 |
| Reddit 글·댓글 | 올라온 때 | 올라온 직후. `created_et` 로 자름 |
| Reddit `score` / `n_comments` | 글이 올라온 때 | **36시간 뒤**. 쓰지 말 것 |

1시간봉이 함정임. 09:30 봉은 09:30~10:30 구간이라 10:30 이 돼야 채워짐.
시작 시각으로 자르면 09:30 에 그 봉을 이미 아는 셈이 됨.

기준일 일봉은 16:00 에 마감돼서 보이고, 대상일 일봉은 안 보임. 정답은 `day.y` 로만 볼 수 있음. `known_at` 이 16:00 이라 장중 데이터에는 안 섞임.

In [5]:
d = day.daily(days=5)
print('기준일 일봉이 보이나:', (d.date_et == day.date).any())
print('대상일 일봉이 보이나:', (d.date_et == day.target).any())
day.y.head(3)

기준일 일봉이 보이나: True
대상일 일봉이 보이나: False


,symbol,date,prev_close,close,ret_pct,label
0,ADI,2026-09-02,354.589996,355.640015,0.296122,2
1,AMD,2026-09-02,459.750000,456.299988,-0.750410,1
2,AMGN,2026-09-02,438.119995,442.839996,1.077331,3


In [6]:
ds.balance()

,이름,비율%
0,급하락,7.4
1,하락,24.1
2,보합,32.9
3,상승,27.4
4,급상승,8.3


## 2. 기간 나누기

`ds.split(날짜)` 하나면 됨. 기준일 앞이 학습, 기준일부터가 평가임.

**과거로 미래를 예측하는 순서**를 지켜야 함. 무작위로 섞으면
내일 데이터로 오늘을 학습하는 꼴이 됨.

In [7]:
train, test = ds.split('2026-06-01')
print(f'학습 {len(train)}일 {train[0].date:%Y-%m-%d}~{train[-1].date:%Y-%m-%d}')
print(f'평가 {len(test)}일 {test[0].date:%Y-%m-%d}~{test[-1].date:%Y-%m-%d}')

# 노트북에서는 빠르게 돌려 보는 게 목적이라 구간을 줄여서 씀.
# 실제로 제출할 때는 아래 두 줄을 지우고 전체 구간으로 학습하면 됨.
train, test = train[-120:], test[:20]
print(f'\n[축소] 학습 {len(train)}일 {train[0].date:%m-%d}~{train[-1].date:%m-%d}'
      f' | 평가 {len(test)}일 {test[0].date:%m-%d}~{test[-1].date:%m-%d}')

학습 449일 2024-08-13~2026-05-28
평가 73일 2026-05-29~2026-09-11

[축소] 학습 120일 12-04~05-28 | 평가 20일 05-29~06-26


In [8]:
def walk_forward(ds, starts):
    """기준일을 옮겨 가며 학습 구간을 늘림."""
    folds = []
    for i, on in enumerate(starts):
        end = starts[i + 1] if i + 1 < len(starts) else None
        tr, rest = ds.split(on)
        te = [d for d in rest if end is None or d.date < pd.Timestamp(end)]
        folds.append((tr, te))
        print(f'fold {i}  학습 {len(tr):>3}일 (~{tr[-1].date:%Y-%m-%d})'
              f'  평가 {len(te):>3}일 ({te[0].date:%Y-%m-%d}~{te[-1].date:%Y-%m-%d})')
    return folds

folds = walk_forward(ds, ['2026-03-01', '2026-05-01', '2026-07-01'])

fold 0  학습 386일 (~2026-02-26)  평가  44일 (2026-02-27~2026-04-30)


fold 1  학습 429일 (~2026-04-29)  평가  42일 (2026-04-30~2026-06-30)


fold 2  학습 470일 (~2026-06-29)  평가  52일 (2026-06-30~2026-09-11)


In [9]:
# rolling — 학습 구간 길이를 고정함. 예전 시장은 지금과 다르다고 볼 때 씀.
for i, (tr, te) in enumerate(folds):
    tr = tr[-120:]
    print(f'fold {i}  학습 {len(tr)}일 ({tr[0].date:%Y-%m-%d}~{tr[-1].date:%Y-%m-%d})')

fold 0  학습 120일 (2025-09-05~2026-02-26)
fold 1  학습 120일 (2025-11-05~2026-04-29)
fold 2  학습 120일 (2026-01-06~2026-06-29)


## 3. 기술지표로 feature 만들기

`src/features/` 의 RSI · MACD · 볼린저를 씀.
세 함수 다 **DataFrame** 을 돌려주니 필요한 열만 꺼내 쓰면 됨.

지표를 계산하려면 과거 며칠이 필요하니 `day.daily(days=...)` 로 넉넉히 받아 옴.

In [10]:
c = day.daily(symbols=['AAPL'] if 'AAPL' in day.symbols else [day.symbols[0]],
              days=240).sort_values('date_et')['close'].reset_index(drop=True)

print('rsi      ', list(rsi(c).columns))
print('macd     ', list(macd(c).columns))
print('bollinger', list(bollinger(c).columns))

bollinger(c).tail(3).round(3)

rsi       ['avg_gain', 'avg_loss', 'rs', 'rsi']
macd      ['macd', 'signal', 'hist', 'macd_over_close']
bollinger ['middle', 'sd', 'upper', 'lower', 'pct_b', 'band_width']


,middle,sd,upper,lower,pct_b,band_width
237,377.402,8.254,393.909,360.895,0.028,0.087
238,377.410,8.238,393.886,360.934,0.037,0.087
239,376.124,9.648,395.421,356.828,-0.058,0.103


In [11]:
LOOKBACK = 120
FEATURES = ['ret1', 'ret5', 'rsi14', 'macd_hist', 'pct_b', 'band_width',
            'vol_ratio', 'pre_ret']


def one_symbol(g):
    """종목 하나의 일봉에서 feature 한 줄을 뽑음. 마지막 날 기준."""
    close = g['close'].reset_index(drop=True)
    if len(close) < 30:
        return None
    m, b, r = macd(close), bollinger(close, n=20), rsi(close, n=14)
    return {
        'ret1': close.pct_change().iloc[-1],
        'ret5': close.iloc[-1] / close.iloc[-6] - 1 if len(close) > 5 else np.nan,
        'rsi14': r['rsi'].iloc[-1],
        'macd_hist': m['hist'].iloc[-1] / close.iloc[-1],   # 가격으로 나눠야 종목끼리 비교됨
        'pct_b': b['pct_b'].iloc[-1],
        'band_width': b['band_width'].iloc[-1],
        'vol_ratio': g['volume'].iloc[-1] / g['volume'].tail(20).mean(),
    }


def build(day, lookback=LOOKBACK):
    """그날 예측에 쓸 feature. 종목마다 한 줄씩."""
    d = day.daily(days=lookback,          # days 는 거래일 기준이라 휴장일은 알아서 건너뜀
                  columns=['symbol', 'date_et', 'close', 'volume'])
    d = d.sort_values(['symbol', 'date_et'])

    rows = []
    for symbol, g in d.groupby('symbol'):
        x = one_symbol(g)
        if x is not None:
            rows.append({'symbol': symbol, **x})
    if not rows:                          # 앞쪽 며칠은 일봉이 모자람
        return pd.DataFrame(columns=['symbol', *FEATURES])
    x = pd.DataFrame(rows)

    # 개장 전 움직임. 09:30 직전까지 나온 시간외 봉을 씀.
    p = day.price(hours=20, columns=['symbol', 'datetime', 'close', 'session'])
    pre = p[p.session == 'pre'].sort_values('datetime').groupby('symbol')['close']
    x['pre_ret'] = x.symbol.map(
        pre.apply(lambda s: s.iloc[-1] / s.iloc[0] - 1 if len(s) > 1 else np.nan))

    x = x[x.symbol.isin(day.symbols)].reset_index(drop=True)
    return x.replace([np.inf, -np.inf], np.nan)


build(day).head().round(4)

,symbol,ret1,ret5,rsi14,macd_hist,pct_b,band_width,vol_ratio,pre_ret
0,ADI,-0.0208,-0.0510,35.7042,-0.0052,-0.0580,0.1026,0.9097,0.0073
1,AMD,-0.0217,-0.0407,43.6826,-0.0015,0.1675,0.1170,0.7840,0.0019
2,AMGN,0.0190,-0.0092,65.8730,-0.0035,0.7271,0.1207,0.9214,0.0001
3,AMZN,-0.0193,-0.0232,46.3191,-0.0057,0.1270,0.0976,0.9213,0.0019
4,AVGO,-0.0032,0.0363,43.3389,-0.0045,0.3289,0.2701,0.9847,0.0036


## 4. 기준선 — 어제 label 따라가기

학습이 없음. 전날 수익률을 보고 같은 label 을 찍음.
**이걸 못 이기면 모델이 아무것도 못 배운 거임.**

In [12]:
from src import label_of


def baseline_predict(day):
    """전날 등락률을 그대로 label 로 바꿔서 찍음."""
    d = day.daily(days=10, columns=['symbol', 'date_et', 'ret'])
    last = d.sort_values('date_et').groupby('symbol')['ret'].last() * 100

    rows = []
    for symbol in day.symbols:
        r = last.get(symbol)
        label = FLAT if r is None or pd.isna(r) else int(label_of(pd.Series([r]))[0])
        rows.append({'symbol': symbol, 'label': label})
    return pd.DataFrame(rows)


class Baseline:
    """모델은 predict(day) 만 있으면 됨."""

    def predict(self, day):
        return baseline_predict(day)


# 채점할 때와 같은 함수로 평가함
r = evaluate(Baseline(), test)

평가 20일 (2026-05-29 ~ 2026-06-26)


  20/20일


예측 0.6초

점수         -0.1220   1 = 완벽, 0 = 우연 수준
정확도       0.220
방향 적중    0.375   보합 제외
급등락 포착  0.174   실제 급등락 중 맞힌 비율
급등락 적중  0.173   급등락이라 찍은 것 중 맞은 비율
건수         1,000

예측   급하락  하락  보합  상승  급상승
실제                       
급하락   24  37  27  24   32
하락    26  62  76  55   38
보합    34  58  45  64   20
상승    26  66  51  63   29
급상승   34  40  18  25   26


## 5. 랜덤 포레스트

feature 를 날짜별로 쌓아서 표 하나로 만들고 분류기에 넣음.
하루에 종목 수만큼 행이 생김.

> 구간을 줄였기 때문에 금방 끝남.

In [13]:
from sklearn.ensemble import RandomForestClassifier


def make_table(days, with_label=True):
    rows = []
    for i, d in enumerate(days, 1):
        x = build(d)
        if not len(x):
            continue
        if with_label:
            x = x.merge(d.y[['symbol', 'label']], on='symbol')
        x['date'] = d.date
        rows.append(x)
        if i % 100 == 0:
            print(f'  {i}/{len(days)}일')
    return pd.concat(rows, ignore_index=True)


tr = make_table(train).dropna(subset=FEATURES)
te = make_table(test).dropna(subset=FEATURES)
print(tr.shape, te.shape)

  100/120일


(5929, 11) (1000, 11)


In [14]:
clf = RandomForestClassifier(n_estimators=300, max_depth=8,
                             min_samples_leaf=50, n_jobs=-1, random_state=0)
clf.fit(tr[FEATURES], tr['label'].astype(int))

class Forest:
    def __init__(self, clf):
        self.clf = clf

    def predict(self, day):
        x = build(day)
        out = pd.DataFrame({'symbol': day.symbols}).merge(x, on='symbol', how='left')
        ok = out[FEATURES].notna().all(axis=1)

        # feature 를 만들지 못한 종목은 보합으로 둠
        label = np.full(len(out), FLAT, dtype=int)
        if ok.any():
            label[ok.values] = self.clf.predict(out.loc[ok, FEATURES])
        return pd.DataFrame({'symbol': out.symbol, 'label': label})


forest = Forest(clf)
r = evaluate(forest, test)

평가 20일 (2026-05-29 ~ 2026-06-26)


  20/20일


예측 6.3초

점수         +0.0475   1 = 완벽, 0 = 우연 수준
정확도       0.259
방향 적중    0.304   보합 제외
급등락 포착  0.056   실제 급등락 중 맞힌 비율
급등락 적중  0.348   급등락이라 찍은 것 중 맞은 비율
건수         1,000

예측   급하락  하락   보합  상승  급상승
실제                        
급하락    5  38   49  40   12
하락     4  52  139  57    5
보합     1  37  129  52    2
상승     1  30  140  62    2
급상승    3  26   40  63   11


In [15]:
pd.Series(clf.feature_importances_, index=FEATURES).sort_values(ascending=False).round(3)

pre_ret       0.249
band_width    0.194
ret5          0.124
ret1          0.123
vol_ratio     0.110
macd_hist     0.072
rsi14         0.070
pct_b         0.058
dtype: float64

## 6. LSTM 한 층

표 한 줄로 줄이지 않고 **과거 N일을 통째로** 넣음.
`day.daily(days=...)` 가 구간을 주니 sequence 도 그냥 만들어짐.

In [16]:
import torch
import torch.nn as nn

SEQ = 20
COLS = ['ret', 'range', 'vol_z']


def sequences(day, seq=SEQ):
    """종목별 (seq, feature수) 배열. 마지막 시점이 예측 직전 거래일."""
    d = day.daily(days=seq * 3,
                  columns=['symbol', 'date_et', 'high', 'low', 'close',
                           'volume', 'ret'])
    if not len(d):
        return [], np.zeros((0, seq, len(COLS)), dtype=np.float32)
    d = d.sort_values(['symbol', 'date_et']).copy()
    d['range'] = (d['high'] - d['low']) / d['close']
    v = d.groupby('symbol')['volume']
    d['vol_z'] = (d['volume'] - v.transform('mean')) / (v.transform('std') + 1e-9)
    d[COLS] = d[COLS].replace([np.inf, -np.inf], np.nan).fillna(0.0)

    syms, arrs = [], []
    for symbol, g in d.groupby('symbol'):
        if symbol not in day.symbols or len(g) < seq:
            continue
        syms.append(symbol)
        arrs.append(g[COLS].to_numpy(dtype=np.float32)[-seq:])
    if not arrs:
        return [], np.zeros((0, seq, len(COLS)), dtype=np.float32)
    return syms, np.stack(arrs)


class Net(nn.Module):
    """LSTM 한 층에 선형 하나. 마지막 시점의 은닉 상태만 씀."""

    def __init__(self, n_features=len(COLS), hidden=32, n_class=5):
        super().__init__()
        self.lstm = nn.LSTM(n_features, hidden, batch_first=True)
        self.head = nn.Linear(hidden, n_class)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.head(out[:, -1])

In [17]:
def build_set(days, seq=SEQ):
    X, y = [], []
    for d in days:
        syms, arr = sequences(d, seq)
        if not syms:
            continue
        lab = d.y.set_index('symbol')['label']
        keep = [j for j, s in enumerate(syms) if s in lab.index]
        if not keep:
            continue
        X.append(arr[keep])
        y.append(lab.loc[[syms[j] for j in keep]].to_numpy(dtype=np.int64))
    return np.concatenate(X), np.concatenate(y)


torch.manual_seed(0)
Xtr, ytr = build_set(train)
Xte, yte = build_set(test)
print(Xtr.shape, np.bincount(ytr))

(6000, 20, 3) [ 486 1489 1865 1578  582]


In [18]:
net = Net()
opt = torch.optim.Adam(net.parameters(), lr=1e-3)
lossf = nn.CrossEntropyLoss()
Xt, yt = torch.from_numpy(Xtr), torch.from_numpy(ytr)

for ep in range(1, 6):
    perm = torch.randperm(len(Xt))
    total = 0.0
    for i in range(0, len(perm), 256):
        idx = perm[i:i + 256]
        opt.zero_grad()
        loss = lossf(net(Xt[idx]), yt[idx])
        loss.backward()
        opt.step()
        total += loss.item() * len(idx)
    print(f'epoch {ep:>2}  loss {total / len(perm):.4f}')

epoch  1  loss 1.6137
epoch  2  loss 1.5212


epoch  3  loss 1.4890
epoch  4  loss 1.4834
epoch  5  loss 1.4825


In [19]:
net.eval()


class Lstm:
    def __init__(self, net):
        self.net = net

    def predict(self, day):
        syms, arr = sequences(day)
        label = dict.fromkeys(day.symbols, FLAT)
        if syms:
            with torch.no_grad():
                out = self.net(torch.from_numpy(arr)).argmax(1).numpy()
            label.update(dict(zip(syms, out)))
        return pd.DataFrame({'symbol': list(label), 'label': list(label.values())})


r = evaluate(Lstm(net), test)

평가 20일 (2026-05-29 ~ 2026-06-26)


  20/20일


예측 2.0초

점수         -0.0097   1 = 완벽, 0 = 우연 수준
정확도       0.224
방향 적중    0.041   보합 제외
급등락 포착  0.000   실제 급등락 중 맞힌 비율
급등락 적중    -     급등락이라 찍은 것 중 맞은 비율
건수         1,000

예측   급하락  하락   보합  상승  급상승
실제                        
급하락    0   0  129  15    0
하락     0   0  239  18    0
보합     0   0  204  17    0
상승     0   0  215  20    0
급상승    0   0  131  12    0


여기까지 한 걸 `src/model.py` 에 옮기면 제출 준비가 끝남.

아래 코드는 `example/sample_model.py` 에 정리해 두었다.

`src/model.py` 가 갖춰야 하는 건 두 개다.

```python
def load_model(): ...            # 학습해 둔 모델을 돌려줌
class Model:
    def predict(self, day): ...  # symbol / label column
```

학습 방법이나 위치에는 제약이 없음. 결과를 저장해 두고 `load_model()` 에서
읽어오기만 하면 됨. 채점할 때는 학습을 다시 돌리지 않음.

> 클래스를 통째로 pickle하면 안 됨. 노트북에서 정의한 클래스는 다른 파일에서
> 읽을 때 `Can't get attribute` 로 깨짐. 학습 결과(`clf`, 가중치, 스케일러)만
> 저장하고 클래스는 `src/model.py` 에 둠.

랜덤 포레스트를 예로 들면 이렇다.

In [20]:
import pickle

ARTIFACT = ARTIFACTS / 'model.pkl'

# 학습 결과만 저장함. 클래스를 통째로 저장하면 다른 파일에서 못 읽음
with open(ARTIFACT, 'wb') as f:
    pickle.dump(clf, f)

print('저장', ARTIFACT)

저장 /home/yeop/Music/business-analytics/finance/project/artifacts/model.pkl


`src/model.py` 를 고쳤으면 출력 형식부터 확인함.

```bash
python main.py check   --split 2026-06-01    # 규격만 검사
python main.py eval    --split 2026-06-01    # 점수
python main.py predict --date  2026-09-10    # 하루 예측
```

`check` 를 통과시켜 두면 채점 도중에 멈출 일이 없음.

In [21]:
# src/model.py 가 이런 모습이 됨. 여기서는 확인만 함
with open(ARTIFACT, 'rb') as f:
    loaded_clf = pickle.load(f)


class Model:
    """`src/model.py` 에 이대로 넣으면 됨."""

    def __init__(self, clf):
        self.clf = clf

    def predict(self, day):
        x = build(day)
        out = pd.DataFrame({'symbol': day.symbols})
        x = out.merge(x, on='symbol', how='left')
        ok = x[FEATURES].notna().all(axis=1)

        # feature 를 만들지 못한 종목은 보합으로 둠. 빠뜨리면 형식 오류가 남
        label = np.full(len(x), FLAT, dtype=int)
        if ok.any():
            label[ok.values] = self.clf.predict(x.loc[ok, FEATURES])
        return pd.DataFrame({'symbol': x.symbol, 'label': label})


model = Model(loaded_clf)

# 제출 전에 형식부터 확인함
check(model, test, n=3)

# 그리고 채점할 때와 같은 함수로 점수를 냄
r = evaluate(model, test)

3일 검사



  2026-06-01   50종목  분포 {0: 1, 1: 11, 2: 23, 3: 12, 4: 3}


  2026-06-16   50종목  분포 {1: 10, 2: 32, 3: 8}


  2026-06-17   50종목  분포 {0: 2, 1: 16, 2: 18, 3: 12, 4: 2}

형식 확인 완료
평가 20일 (2026-05-29 ~ 2026-06-26)


  20/20일


예측 6.4초

점수         +0.0475   1 = 완벽, 0 = 우연 수준
정확도       0.259
방향 적중    0.304   보합 제외
급등락 포착  0.056   실제 급등락 중 맞힌 비율
급등락 적중  0.348   급등락이라 찍은 것 중 맞은 비율
건수         1,000

예측   급하락  하락   보합  상승  급상승
실제                        
급하락    5  38   49  40   12
하락     4  52  139  57    5
보합     1  37  129  52    2
상승     1  30  140  62    2
급상승    3  26   40  63   11
